## Function 2: Read Raster Windows and Calculate NDVI 🌿

In this notebook, you'll learn how to build the `windowed_read()` and `calculate_ndvi()` functions step by step. This is where you read only the pixels you need from cloud-hosted raster imagery and calculate NDVI from red and near-infrared bands.

### 🎯 What This Function Does
- Read raster subsets using a bounding box
- Convert geographic coordinates into raster windows
- Return pixel values and a window transform
- Calculate NDVI from red and NIR bands
- Visualize raster subsets and NDVI results

### 🔧 Function Signature
```python
def windowed_read(url, bbox):
    """
    Args:
        url (str): Raster asset URL
        bbox (list): [west, south, east, north]
    
    Returns:
        tuple: A tuple containing pixel values and a window transform
    """
```

```python
def calculate_ndvi(red_pixels, nir_pixels):
    """
    Args:
        red_pixels (numpy.ndarray): Red band pixel values
        nir_pixels (numpy.ndarray): Near-infrared band pixel values
    
    Returns:
        numpy.ndarray: NDVI values
    """
```

### 📍 Where This Function Goes:
**Target File**: `src/rasterio_basics.py`  
**Function Names**: `windowed_read()`, `calculate_ndvi()`  
**Replace**: The placeholder functions with your working code

---

### 📚 Step 1: Import Libraries and Search for One Scene

Let's import the libraries needed to read cloud-hosted rasters and work with image arrays.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.plot import show, reshape_as_image
from pyproj import Transformer
from pystac_client import Client
import planetary_computer

stac_url = "https://planetarycomputer.microsoft.com/api/stac/v1"

bbox = [-110.74772571639987, 32.270431012618026, -110.70996215904789, 32.29386169894274]
date_range = "2021-07-05/2021-08-02"

catalog = Client.open(stac_url)
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=date_range,
    query={"eo:cloud_cover": {"lt": 5}}
)
items = list(search.items())
item = items[0]

print(f"✅ Found {len(items)} items")
print(f"✅ Using item: {item.id}")

### 🪟 Step 2: Understanding Windowed Reads

**💡 This will be used in our function!**

In [ ]:
red_url = item.assets["B04"].href
signed_red_url = planetary_computer.sign(red_url)

with rasterio.open(signed_red_url) as src:
    print(f"Width: {src.width}")
    print(f"Height: {src.height}")
    print(f"CRS: {src.crs}")
    print(f"Transform: {src.transform}")

### 🌍 Step 3: Convert the Bounding Box to the Raster CRS

**💡 This will be used in our function!**

In [ ]:
with rasterio.open(signed_red_url) as src:
    transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
    left, bottom = transformer.transform(bbox[0], bbox[1])
    right, top = transformer.transform(bbox[2], bbox[3])
    
    print(f"Left: {left:.2f}")
    print(f"Bottom: {bottom:.2f}")
    print(f"Right: {right:.2f}")
    print(f"Top: {top:.2f}")

### 🔢 Step 4: Convert Geographic Coordinates to Pixel Coordinates

**💡 This will be used in our function!**

In [ ]:
with rasterio.open(signed_red_url) as src:
    transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
    left, bottom = transformer.transform(bbox[0], bbox[1])
    right, top = transformer.transform(bbox[2], bbox[3])
    
    row_start, col_start = src.index(left, top)
    row_stop, col_stop = src.index(right, bottom)
    
    print(f"Top-left pixel: ({row_start}, {col_start})")
    print(f"Bottom-right pixel: ({row_stop}, {col_stop})")
    print(f"Rows: {row_stop - row_start}")
    print(f"Cols: {col_stop - col_start}")

### 📥 Step 5: Read a Raster Window

Now let's read only the subset of pixels inside our area of interest.

In [ ]:
with rasterio.open(signed_red_url) as src:
    transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
    left, bottom = transformer.transform(bbox[0], bbox[1])
    right, top = transformer.transform(bbox[2], bbox[3])
    
    row_start, col_start = src.index(left, top)
    row_stop, col_stop = src.index(right, bottom)
    
    window = rasterio.windows.Window.from_slices(
        (row_start, row_stop),
        (col_start, col_stop)
    )
    
    red_pixels = src.read(1, window=window)
    window_transform = src.window_transform(window)

print(f"✅ Shape: {red_pixels.shape}")
print(f"✅ Dtype: {red_pixels.dtype}")
print(f"✅ Min: {red_pixels.min()}")
print(f"✅ Max: {red_pixels.max()}")

### 🌿 Step 6: Calculate NDVI

**💡 This will be used in our function!**

In [ ]:
nir_url = item.assets["B08"].href
signed_nir_url = planetary_computer.sign(nir_url)

with rasterio.open(signed_nir_url) as src:
    transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
    left, bottom = transformer.transform(bbox[0], bbox[1])
    right, top = transformer.transform(bbox[2], bbox[3])
    
    row_start, col_start = src.index(left, top)
    row_stop, col_stop = src.index(right, bottom)
    
    window = rasterio.windows.Window.from_slices(
        (row_start, row_stop),
        (col_start, col_stop)
    )
    
    nir_pixels = src.read(1, window=window)

ndvi = (nir_pixels.astype(float) - red_pixels.astype(float)) / (
    nir_pixels.astype(float) + red_pixels.astype(float)
)

print(f"✅ NDVI shape: {ndvi.shape}")
print(f"✅ NDVI min: {ndvi.min():.3f}")
print(f"✅ NDVI max: {ndvi.max():.3f}")
print(f"✅ NDVI mean: {ndvi.mean():.3f}")

### 🖼️ Step 7: Visualize Raster Data and NDVI

Let's compare the raster subset and the NDVI result.

In [ ]:
rgb_url = item.assets["visual"].href
signed_rgb_url = planetary_computer.sign(rgb_url)

with rasterio.open(signed_rgb_url) as src:
    transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
    left, bottom = transformer.transform(bbox[0], bbox[1])
    right, top = transformer.transform(bbox[2], bbox[3])
    
    row_start, col_start = src.index(left, top)
    row_stop, col_stop = src.index(right, bottom)
    
    window = rasterio.windows.Window.from_slices(
        (row_start, row_stop),
        (col_start, col_stop)
    )
    
    rgb_pixels = src.read([1, 2, 3], window=window)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

show(rgb_pixels, transform=window_transform, ax=ax1)
ax1.set_title("RGB Subset")

im = ax2.imshow(ndvi, cmap="RdYlGn", vmin=-1, vmax=1)
ax2.set_title("NDVI")
ax2.axis("off")
plt.colorbar(im, ax=ax2, shrink=0.8)

plt.tight_layout()
plt.show()

### 🧩 Step 8: Building the Complete Functions

Now let's put everything together into reusable functions. This is what you will implement in `src/rasterio_basics.py`.

In [ ]:
import rasterio
import planetary_computer
import numpy as np
from pyproj import Transformer

def windowed_read(url, bbox):
    """
    Read a raster subset from a cloud-hosted asset.
    
    Args:
        url: Raster asset URL
        bbox: [west, south, east, north]
        
    Returns:
        Tuple of (pixels, transform)
    """
    signed_url = planetary_computer.sign(url)
    
    with rasterio.open(signed_url) as src:
        transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        left, bottom = transformer.transform(bbox[0], bbox[1])
        right, top = transformer.transform(bbox[2], bbox[3])
        
        row_start, col_start = src.index(left, top)
        row_stop, col_stop = src.index(right, bottom)
        
        window = rasterio.windows.Window.from_slices(
            (row_start, row_stop),
            (col_start, col_stop)
        )
        
        pixels = src.read(1, window=window)
        transform = src.window_transform(window)
        
        return pixels, transform

def calculate_ndvi(red_pixels, nir_pixels):
    """
    Calculate NDVI from red and NIR band arrays.
    """
    red = red_pixels.astype(float)
    nir = nir_pixels.astype(float)
    return (nir - red) / (nir + red)

### ✨ Step 9: Test Your Functions

Let's test our complete functions with red and NIR scene assets.

In [ ]:
# Test 1: Windowed read
print("Test 1: Windowed read")
red_pixels_test, transform_test = windowed_read(item.assets["B04"].href, bbox)
print(f"✅ Shape: {red_pixels_test.shape}")
print(f"✅ Returned transform: {transform_test}")
print()

# Test 2: NIR read
print("Test 2: NIR read")
nir_pixels_test, _ = windowed_read(item.assets["B08"].href, bbox)
print(f"✅ Shape: {nir_pixels_test.shape}")
print()

# Test 3: NDVI calculation
print("Test 3: NDVI calculation")
ndvi_test = calculate_ndvi(red_pixels_test, nir_pixels_test)
print(f"✅ Shape: {ndvi_test.shape}")
print(f"✅ Mean NDVI: {ndvi_test.mean():.3f}")
print(f"✅ Min NDVI: {ndvi_test.min():.3f}")
print(f"✅ Max NDVI: {ndvi_test.max():.3f}")
print()

print("🎉 All tests completed!")

### 🧪 Step 10: Verify with Pytest

Test your entire functions to verify your implementation in `src/rasterio_basics.py` by running the following line in the terminal

```bash
uv run pytest tests/ -k "TestWindowedRead or TestCalculateNDVI" -v
```

**⚠️ IMPORTANT: Make sure this passes before you move on!**

---

### 🔑 Key Learning Points

- **Cloud-Optimized GeoTIFFs (COGs)** support efficient raster access over the web
- **Windowed reads** download only the pixels needed for the analysis area
- A raster **transform** connects pixel locations to geographic coordinates
- **Red** and **NIR** bands are commonly used to calculate NDVI
- **NDVI** is computed as `(NIR - Red) / (NIR + Red)`
- NDVI is useful for identifying vegetation patterns and plant health